In [1]:
# Import des librairies

import pandas as pd
import numpy as np
from datetime import datetime
import openpyxl as op
import sklearn as sk
from sklearn.linear_model import LinearRegression


# Charger les DF des différents pages

DF : projects_plans, actual_duration, project_type, locations, country_profiles  
- Vérifier que les headers sont OK  
- afficher les types de données  
- afficher actual_duration, projects_plans


In [2]:
# Chargement des donnees
from pathlib import Path

excel_path = Path("data") / "Donnees+Sanitoral.xlsx"
if not excel_path.exists():
    data_dir = Path("data")
    available = [p.name for p in data_dir.glob("*.xlsx")] if data_dir.exists() else []
    raise FileNotFoundError(f"Fichier introuvable: {excel_path}. Disponibles: {available}")

xls = pd.ExcelFile(excel_path)
print("Sheets disponibles:", xls.sheet_names)

headers = {
    "Projects_plans": 2,
    "Project type": 3,
    "Actual_Costs": 3,
    "Actual_Duration": 5,
    "Actual_Delivrable": 3,
    "Projects_Locations": 1,
    "Country_Profiles": 1,
}

missing = set(headers) - set(xls.sheet_names)
if missing:
    print("Sheets manquants:", missing)

dfs = {
    sheet: pd.read_excel(excel_path, sheet_name=sheet, header=header)
    for sheet, header in headers.items()
    if sheet in xls.sheet_names
}

projects_plans = dfs.get("Projects_plans")
projects_type = dfs.get("Project type")
actual_costs = dfs.get("Actual_Costs")
actual_duration = dfs.get("Actual_duration")
actual_delivrable = dfs.get("Actual_Delivrable")
projects_locations = dfs.get("Projects_Locations")
country_profiles = dfs.get("Country_profiles")

Sheets disponibles: ['Projects_plans', 'Project type', 'Actual_Costs', 'Actual_Duration', 'Actual_Delivrable', 'Projects_Locations', 'Country_Profiles']


In [3]:
# Apercu des onglets et tailles
for name, df in dfs.items():
    print(f"{name}: {df.shape[0]} lignes, {df.shape[1]} colonnes")

Projects_plans: 520 lignes, 6 colonnes
Project type: 104 lignes, 2 colonnes
Actual_Costs: 520 lignes, 3 colonnes
Actual_Duration: 520 lignes, 3 colonnes
Actual_Delivrable: 520 lignes, 3 colonnes
Projects_Locations: 104 lignes, 2 colonnes
Country_Profiles: 52 lignes, 3 colonnes


# Exploration des sheets et headers

In [4]:
# Routine d'exploration des sheets et headers Excel
def explore_excel_heads(excel_path, max_rows=2, max_cols=10):
    """
    Explore all sheets in Excel file to find the correct header row and column count.
    Suggests header row (first non-empty row below metadata).
    """
    xls = pd.ExcelFile(excel_path)
    results = {}
    
    for sheet in xls.sheet_names:
        print(f"\n='-- {sheet} --'")
        # Read without header to see all rows
        df_raw = pd.read_excel(excel_path, sheet_name=sheet, header=None)
        print(f"Shape: {df_raw.shape}")
        print(f"Non-null rows (first 5):")
        
        # Find first row with data
        for i, row in df_raw.head(8).iterrows():
            non_null = row.notna().sum()
            print(f"  Row {i}: {non_null} colonnes - {row.head(max_cols).to_list()}")
        
        # Try to detect header (first row with many values)
        header_row = 0
        for i in range(min(10, len(df_raw))):
            if df_raw.iloc[i].notna().sum() > 2:
                header_row = i
                break
        
        results[sheet] = header_row
        print(f"\nSuggested header row: {header_row}")
    
    return results

# Executer l'exploration
suggested_headers = explore_excel_heads(excel_path)


='-- Projects_plans --'
Shape: (523, 6)
Non-null rows (first 5):
  Row 0: 1 colonnes - ['In this tab are the initial forecast of each project launched by Sanitoral', nan, nan, nan, nan, nan]
  Row 1: 0 colonnes - [nan, nan, nan, nan, nan, nan]
  Row 2: 6 colonnes - ['Project ID', 'Phase', 'Start Date', 'Planned_Duration', 'Planned_Cost', 'Planned_Delivrable']
  Row 3: 6 colonnes - [1, 'Phase 1 - Planning', datetime.datetime(2018, 1, 16, 0, 0), 196, 50000, 10]
  Row 4: 6 colonnes - [1, 'Phase 2 - Initiation', datetime.datetime(2018, 1, 31, 0, 0), 15, 100000, 17]
  Row 5: 6 colonnes - [1, 'Phase 3 - Implementation', datetime.datetime(2018, 4, 17, 0, 0), 197, 150000, 26]
  Row 6: 6 colonnes - [1, 'Phase 4 - Manufacturing', datetime.datetime(2018, 10, 10, 0, 0), 61, 450000, 23]
  Row 7: 6 colonnes - [2, 'Phase 1 - Planning', datetime.datetime(2018, 1, 30, 0, 0), 16, 100000, 6]

Suggested header row: 2

='-- Project type --'
Shape: (108, 2)
Non-null rows (first 5):
  Row 0: 1 colonnes - ['

In [5]:
# Afficher le dictionnaire suggere pour copier-coller
print("\n=== Headers suggerees ===")
print("headers = {")
for sheet, header in suggested_headers.items():
    print(f'    \"{sheet}\": {header},')
print("}")


=== Headers suggerees ===
headers = {
    "Projects_plans": 2,
    "Project type": 0,
    "Actual_Costs": 3,
    "Actual_Duration": 5,
    "Actual_Delivrable": 3,
    "Projects_Locations": 0,
    "Country_Profiles": 1,
}


# Verification de la qualite des donnees

In [6]:
def check_data_quality(dfs_dict, verbose=True):
    """
    Verifie les anomalies dans les dataframes:
    - Colonnes/lignes vides
    - Valeurs manquantes
    - Doublons
    """
    quality_report = {}
    
    for sheet_name, df in dfs_dict.items():
        if df is None:
            print(f"\n⚠️  {sheet_name}: DataFrame est None")
            continue
        
        report = {
            'shape': df.shape,
            'empty_cols': [],
            'mostly_empty_cols': [],
            'null_count': df.isnull().sum(),
            'null_pct': (df.isnull().sum() / len(df) * 100).round(2),
            'empty_rows': (df.isnull().sum(axis=1) == len(df)).sum(),
            'duplicates': df.duplicated().sum(),
        }
        
        # Identifier colonnes vides
        for col in df.columns:
            if df[col].isnull().all():
                report['empty_cols'].append(col)
            elif df[col].isnull().sum() / len(df) > 0.8:
                report['mostly_empty_cols'].append(col)
        
        quality_report[sheet_name] = report
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"Sheet: {sheet_name}")
            print(f"{'='*60}")
            print(f"  Shape: {report['shape']}")
            print(f"  Lignes vides: {report['empty_rows']}")
            print(f"  Doublons: {report['duplicates']}")
            print(f"  Colonnes completement vides: {report['empty_cols']}")
            print(f"  Colonnes >80% vides: {report['mostly_empty_cols']}")
            print(f"\n  Valeurs manquantes par colonne:")
            for col, count in report['null_count'].items():
                pct = report['null_pct'][col]
                if count > 0:
                    print(f"    {col}: {count} ({pct}%)")
    
    return quality_report

# Executer la verification
quality_report = check_data_quality(dfs)


Sheet: Projects_plans
  Shape: (520, 6)
  Lignes vides: 0
  Doublons: 0
  Colonnes completement vides: []
  Colonnes >80% vides: []

  Valeurs manquantes par colonne:

Sheet: Project type
  Shape: (104, 2)
  Lignes vides: 0
  Doublons: 0
  Colonnes completement vides: []
  Colonnes >80% vides: []

  Valeurs manquantes par colonne:

Sheet: Actual_Costs
  Shape: (520, 3)
  Lignes vides: 0
  Doublons: 0
  Colonnes completement vides: []
  Colonnes >80% vides: []

  Valeurs manquantes par colonne:

Sheet: Actual_Duration
  Shape: (520, 3)
  Lignes vides: 0
  Doublons: 0
  Colonnes completement vides: []
  Colonnes >80% vides: []

  Valeurs manquantes par colonne:

Sheet: Actual_Delivrable
  Shape: (520, 3)
  Lignes vides: 0
  Doublons: 0
  Colonnes completement vides: []
  Colonnes >80% vides: []

  Valeurs manquantes par colonne:

Sheet: Projects_Locations
  Shape: (104, 2)
  Lignes vides: 0
  Doublons: 0
  Colonnes completement vides: []
  Colonnes >80% vides: []

  Valeurs manquantes p

In [7]:
# Synthese des anomalies detectees
print("\n" + "="*60)
print("SYNTHESE DES ANOMALIES")
print("="*60)

for sheet_name, report in quality_report.items():
    has_issues = (len(report['empty_cols']) > 0 or 
                  len(report['mostly_empty_cols']) > 0 or
                  report['empty_rows'] > 0 or
                  report['duplicates'] > 0)
    
    if has_issues:
        print(f"\n⚠️  {sheet_name}:", end="")
        issues = []
        if report['empty_cols']:
            issues.append(f"{len(report['empty_cols'])} col(s) vides")
        if report['mostly_empty_cols']:
            issues.append(f"{len(report['mostly_empty_cols'])} col(s) >80% vides")
        if report['empty_rows'] > 0:
            issues.append(f"{report['empty_rows']} lignes vides")
        if report['duplicates'] > 0:
            issues.append(f"{report['duplicates']} doublons")
        print(" ".join(issues))
    else:
        print(f"\n✓ {sheet_name}: OK")


SYNTHESE DES ANOMALIES

✓ Projects_plans: OK

✓ Project type: OK

✓ Actual_Costs: OK

✓ Actual_Duration: OK

✓ Actual_Delivrable: OK

✓ Projects_Locations: OK

✓ Country_Profiles: OK



# Export des données nettoyées


In [8]:

# Export des dataframes en fichier Excel
output_path = Path("data") / "DonneesSanitoralNet.xlsx"

# Creer un writer Excel
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in dfs.items():
        if df is not None:
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"✓ {sheet_name}: {df.shape[0]} lignes, {df.shape[1]} colonnes")

print(f"\n✓ Fichier export: {output_path}")
print(f"✓ Chemin absolu: {output_path.absolute()}")


✓ Projects_plans: 520 lignes, 6 colonnes
✓ Project type: 104 lignes, 2 colonnes
✓ Actual_Costs: 520 lignes, 3 colonnes
✓ Actual_Duration: 520 lignes, 3 colonnes
✓ Actual_Delivrable: 520 lignes, 3 colonnes
✓ Projects_Locations: 104 lignes, 2 colonnes
✓ Country_Profiles: 52 lignes, 3 colonnes

✓ Fichier export: data\DonneesSanitoralNet.xlsx
✓ Chemin absolu: c:\Users\feria\Documents\P7demo\data\DonneesSanitoralNet.xlsx


# Jointures

Joindre projects_plans et actual_duration dans un df, par Project ID et Phase  
Attention : vérifier la jointure


In [9]:

# Standardisation des identifiants Project avec auto-increment

# Mapping des colonnes clés dans chaque table
col_mapping = {
    "Projects_plans": "Project ID",
    "Project type": "Project ID",
    "Actual_Costs": "Proj_ID",
    "Actual_Duration": "Project",
    "Actual_Delivrable": "ID",
    "Projects_Locations": "Project ID",
    "Country_Profiles": None,  # Pas de colonne clé
}

# Creer un dictionnaire unique de tous les projets avec auto-increment
all_projects = set()
for sheet_name, col_name in col_mapping.items():
    if col_name and sheet_name in dfs and dfs[sheet_name] is not None:
        all_projects.update(dfs[sheet_name][col_name].dropna().unique())

# Trier et creer un mapping Project -> Project_ID (auto-increment)
project_mapping = {proj: idx + 1 for idx, proj in enumerate(sorted(all_projects))}
print(f"✓ {len(project_mapping)} projets uniques identifies")
print(f"  Sample mapping: {list(project_mapping.items())[:5]}")

# Ajouter la colonne Project_ID a tous les dataframes
for sheet_name, col_name in col_mapping.items():
    if sheet_name in dfs and dfs[sheet_name] is not None:
        df = dfs[sheet_name]
        
        if col_name:
            # Mapper la colonne existante vers Project_ID
            df['Project_ID'] = df[col_name].map(project_mapping)
        else:
            # Pour country_profiles, creer une colonne vide
            df['Project_ID'] = np.nan
        
        print(f"✓ {sheet_name}: Project_ID ajoutee ({df['Project_ID'].notna().sum()} values)")

# Verifier
print("\n=== Colonnes apres standardisation ===")
for sheet_name, df in dfs.items():
    if df is not None:
        print(f"  {sheet_name}: {df.columns.tolist()}")


✓ 104 projets uniques identifies
  Sample mapping: [(np.int64(1), 1), (np.int64(2), 2), (np.int64(3), 3), (np.int64(4), 4), (np.int64(5), 5)]
✓ Projects_plans: Project_ID ajoutee (520 values)
✓ Project type: Project_ID ajoutee (104 values)
✓ Actual_Costs: Project_ID ajoutee (520 values)
✓ Actual_Duration: Project_ID ajoutee (520 values)
✓ Actual_Delivrable: Project_ID ajoutee (520 values)
✓ Projects_Locations: Project_ID ajoutee (104 values)
✓ Country_Profiles: Project_ID ajoutee (0 values)

=== Colonnes apres standardisation ===
  Projects_plans: ['Project ID', 'Phase', 'Start Date', 'Planned_Duration', 'Planned_Cost', 'Planned_Delivrable', 'Project_ID']
  Project type: ['Project ID', 'Project Type', 'Project_ID']
  Actual_Costs: ['Proj_ID', 'Phase', 'Actual_Cost', 'Project_ID']
  Actual_Duration: ['Project', 'Phase', 'Actual_Duration', 'Project_ID']
  Actual_Delivrable: ['ID', 'Phase', 'Actual_Deliverables', 'Project_ID']
  Projects_Locations: ['Project ID', 'Country', 'Project_ID']


In [10]:

# Reasigner les variables apres standardisation
projects_plans = dfs.get("Projects_plans")
projects_type = dfs.get("Project type")
actual_costs = dfs.get("Actual_Costs")
actual_duration = dfs.get("Actual_Duration")  # Note: Actual_Duration (pas actual_duration)
actual_delivrable = dfs.get("Actual_Delivrable")
projects_locations = dfs.get("Projects_Locations")
country_profiles = dfs.get("Country_Profiles")

print("✓ Variables reasignees avec Project_ID")
print(f"  projects_plans: {projects_plans.shape if projects_plans is not None else None}")
print(f"  actual_duration: {actual_duration.shape if actual_duration is not None else None}")


✓ Variables reasignees avec Project_ID
  projects_plans: (520, 7)
  actual_duration: (520, 4)



# Verification des types de donnees

Avant la jointure, on verifie que tous les types de colonnes sont coherents :
- Colonnes numeriques pour les calculs (Duration, Cost, etc.)
- Colonnes texte pour les identifiants (Project_ID)
- Colonnes dates pour les chronologies
- Conversion necessaire des types incompatibles


In [11]:

# Verification des types de donnees avant jointure

def check_dtypes(dfs_dict, verbose=True):
    """
    Verifie les types de donnees dans tous les dataframes.
    Identifie les colonnes numeriques, dates, strings, etc.
    """
    dtype_report = {}
    
    for sheet_name, df in dfs_dict.items():
        if df is None:
            continue
        
        report = {
            'dtypes': df.dtypes.to_dict(),
            'numeric_cols': df.select_dtypes(include=['number']).columns.tolist(),
            'object_cols': df.select_dtypes(include=['object']).columns.tolist(),
            'datetime_cols': df.select_dtypes(include=['datetime']).columns.tolist(),
            'bool_cols': df.select_dtypes(include=['bool']).columns.tolist(),
        }
        
        dtype_report[sheet_name] = report
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"Sheet: {sheet_name}")
            print(f"{'='*60}")
            print(f"  Colonnes numeriques ({len(report['numeric_cols'])}): {report['numeric_cols'][:5]}")
            print(f"  Colonnes texte ({len(report['object_cols'])}): {report['object_cols'][:5]}")
            print(f"  Colonnes date ({len(report['datetime_cols'])}): {report['datetime_cols']}")
            print(f"\n  Tous les types:")
            for col, dtype in report['dtypes'].items():
                print(f"    {col}: {dtype}")
    
    return dtype_report

# Executer la verification
dtype_report = check_dtypes(dfs)



Sheet: Projects_plans
  Colonnes numeriques (5): ['Project ID', 'Planned_Duration', 'Planned_Cost', 'Planned_Delivrable', 'Project_ID']
  Colonnes texte (1): ['Phase']
  Colonnes date (1): ['Start Date']

  Tous les types:
    Project ID: int64
    Phase: str
    Start Date: datetime64[us]
    Planned_Duration: int64
    Planned_Cost: int64
    Planned_Delivrable: int64
    Project_ID: int64

Sheet: Project type
  Colonnes numeriques (2): ['Project ID', 'Project_ID']
  Colonnes texte (1): ['Project Type']
  Colonnes date (0): []

  Tous les types:
    Project ID: int64
    Project Type: str
    Project_ID: int64

Sheet: Actual_Costs
  Colonnes numeriques (3): ['Proj_ID', 'Actual_Cost', 'Project_ID']
  Colonnes texte (1): ['Phase']
  Colonnes date (0): []

  Tous les types:
    Proj_ID: int64
    Phase: str
    Actual_Cost: int64
    Project_ID: int64

Sheet: Actual_Duration
  Colonnes numeriques (3): ['Project', 'Actual_Duration', 'Project_ID']
  Colonnes texte (1): ['Phase']
  Colonn

C:\Users\feria\AppData\Local\Temp\ipykernel_10476\1004739197.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  'object_cols': df.select_dtypes(include=['object']).columns.tolist(),
C:\Users\feria\AppData\Local\Temp\ipykernel_10476\1004739197.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org

In [12]:

# Jointures avec Project_ID standardise

if projects_plans is not None and actual_duration is not None:
    # Jointure sur Project_ID
    df_merged = pd.merge(
        projects_plans,
        actual_duration,
        on='Project_ID',
        how='left',
        suffixes=('_planned', '_actual')
    )
    print(f"✓ Jointure projects_plans + actual_duration")
    print(f"  Shape: {df_merged.shape[0]} lignes, {df_merged.shape[1]} colonnes")
    print(f"  Doubles keys: {df_merged.shape[0] - df_merged['Project_ID'].nunique()}")
else:
    print("⚠️  Impossible de joindre: manque projects_plans ou actual_duration")
    df_merged = None


✓ Jointure projects_plans + actual_duration
  Shape: 2704 lignes, 10 colonnes
  Doubles keys: 2600


In [13]:
# Ajouter la colonne End_date dans projects_plans avec format aligne

if projects_plans is not None:
    # Verifier que les colonnes existent
    if 'Start Date' in projects_plans.columns and 'Planned_Duration' in projects_plans.columns:
        # Convertir Start Date en datetime (format: "16/01/2018  00:00:00")
        projects_plans['Start Date'] = pd.to_datetime(projects_plans['Start Date'], format='%d/%m/%Y  %H:%M:%S', errors='coerce')
        
        # Calculer End_Date = Start Date + Planned_Duration (en jours)
        projects_plans['End_Date'] = projects_plans['Start Date'] + pd.to_timedelta(projects_plans['Planned_Duration'], unit='D')
        
        # Formatter les deux colonnes au même format (avec time)
        date_format = '%d/%m/%Y  %H:%M:%S'
        projects_plans['Start Date'] = projects_plans['Start Date'].dt.strftime(date_format)
        projects_plans['End_Date'] = projects_plans['End_Date'].dt.strftime(date_format)
        
        print("✓ Colonnes Start Date et End_Date realignees")
        print(f"  Format: {date_format}")
        print(f"  Sample:")
        print(projects_plans[['Start Date', 'Planned_Duration', 'End_Date']].head(3))
    else:
        print("⚠️  Colonnes manquantes")
        print(f"  'Start Date' present: {'Start Date' in projects_plans.columns}")
        print(f"  'Planned_Duration' present: {'Planned_Duration' in projects_plans.columns}")
else:
    print("⚠️  projects_plans est None")



✓ Colonnes Start Date et End_Date realignees
  Format: %d/%m/%Y  %H:%M:%S
  Sample:
             Start Date  Planned_Duration              End_Date
0  16/01/2018  00:00:00               196  31/07/2018  00:00:00
1  31/01/2018  00:00:00                15  15/02/2018  00:00:00
2  17/04/2018  00:00:00               197  31/10/2018  00:00:00


# Calculs

Ajouter dans le DF :  
- le retard (j) : Delay = Actual_Duration - Planned_Duration
- Is_Late : 1 = retard, 0 = pas de retard
- Start_Month, Start_Quarter, Start_Year

In [14]:
# Ajouter les colonnes temporelles : Start_Month, Start_Quarter, Start_Year

if df_merged is not None:
    # Convertir Start Date en datetime (format: "2018-01-16" ou "%d/%m/%Y  %H:%M:%S")
    df_merged['Start Date'] = pd.to_datetime(df_merged['Start Date'], errors='coerce')
    
    df_merged['Start_Month'] = df_merged['Start Date'].dt.month
    df_merged['Start_Quarter'] = df_merged['Start Date'].dt.quarter
    df_merged['Start_Year'] = df_merged['Start Date'].dt.year
    
    print("✓ Colonnes calculees ajoutees:")
    print(f"  - Delay et Is_Late: déjà présentes")
    print(f"  - Start_Month, Start_Quarter, Start_Year: extraits de la date de début")
    print(f"\n✓ Sample des colonnes calculees:")
    print(df_merged[['Start Date', 'Planned_Duration', 'Actual_Duration', 'Delay', 'Is_Late', 'Start_Month', 'Start_Quarter', 'Start_Year']].head(3))
else:
    print("⚠️  df_merged est None")

✓ Colonnes calculees ajoutees:
  - Delay et Is_Late: déjà présentes
  - Start_Month, Start_Quarter, Start_Year: extraits de la date de début

✓ Sample des colonnes calculees:


KeyError: "['Delay', 'Is_Late'] not in index"

# Aggrégations

Récupérer par projet (Project ID) :  
- somme Planned_Duration  
- somme Planned_Cost
- somme Planned_Delivrable
- nombre de Phase
- somme Delay
- max Is_Late
- Start_Month
- Start_Year



In [ ]:
#df_projet_features= df.groupby()...
# VOIR IS_late à calculer si phase remplie ou pas
#first correspond d'abord l'année puis après le mois de démarrage pour avoir la min de start_date pour extraire le mois après si les phases sont triées par ordre 

# Graphique histogram

Histogramme des Planned_Duration

In [ ]:
import matplotlib.pyplot as plt
#bins : nbre dde barres projets par mois de démarrage 